In [ ]:
# =============================================================================
# Features Engineer v3 — LaDe-P (Yantai) - Improved Version
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR = "/content/drive/MyDrive/LaDe/LaDe-P"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Save directory:", SAVE_DIR)

Mounted at /content/drive
Save directory: /content/drive/MyDrive/LaDe/LaDe-P


In [ ]:
# ====================== 1. LOAD RAW DATA ======================
df = pd.read_csv("/content/drive/MyDrive/LaDe/Dataset/pickup_yt.csv")

print(f"Original shape: {df.shape}")

Original shape: (1146781, 19)


In [ ]:
# ====================== 2. BASIC CLEANING & TARGET ======================
df["accept_time"] = pd.to_datetime(df["accept_time"], format="%m-%d %H:%M:%S", errors="coerce")
df["pickup_time"] = pd.to_datetime(df["pickup_time"], format="%m-%d %H:%M:%S", errors="coerce")
df["time_window_start"] = pd.to_datetime(df["time_window_start"], format="%m-%d %H:%M:%S", errors="coerce")
df["time_window_end"]   = pd.to_datetime(df["time_window_end"],   format="%m-%d %H:%M:%S", errors="coerce")

df["eta_minutes"] = (df["pickup_time"] - df["accept_time"]).dt.total_seconds() / 60
df = df[df["eta_minutes"] > 0].copy()
df["eta_log"] = np.log1p(df["eta_minutes"])

print(f"After basic cleaning: {df.shape}")

After basic cleaning: (1143846, 21)


In [ ]:
# ====================== 3. HAVERSINE & SPATIAL FEATURES ======================
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    return R * c

# Distance with fallback
df['distance_km'] = np.nan
mask_both = df['accept_gps_lat'].notna() & df['pickup_gps_lat'].notna()
df.loc[mask_both, 'distance_km'] = haversine(
    df.loc[mask_both,'accept_gps_lat'], df.loc[mask_both,'accept_gps_lng'],
    df.loc[mask_both,'pickup_gps_lat'], df.loc[mask_both,'pickup_gps_lng']
)

# Fallback
mask_fallback = df['distance_km'].isna()
df.loc[mask_fallback, 'distance_km'] = haversine(
    df.loc[mask_fallback,'accept_gps_lat'].fillna(df.loc[mask_fallback,'lat']),
    df.loc[mask_fallback,'accept_gps_lng'].fillna(df.loc[mask_fallback,'lng']),
    df.loc[mask_fallback,'lat'], df.loc[mask_fallback,'lng']
)

df['gps_both'] = mask_both.astype(int)
df['gps_accept_missing'] = df['accept_gps_lat'].isna().astype(int)

In [ ]:
# ====================== 4. TIME & RUSH HOUR FEATURES (Cải tiến) ======================
df["accept_hour"] = df["accept_time"].dt.hour
df["accept_minute"] = df["accept_time"].dt.minute
df["day_of_week"] = df["accept_time"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

df['is_morning_rush'] = df['accept_hour'].between(7, 9).astype(int)
df['is_evening_rush'] = df['accept_hour'].between(16, 19).astype(int)
df['is_peak_hour'] = (df['is_morning_rush'] | df['is_evening_rush']).astype(int)
df['is_working_hour'] = df['accept_hour'].between(8, 17).astype(int)

df["hour_sin"] = np.sin(2 * np.pi * df["accept_hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["accept_hour"] / 24)

df["time_to_window_start"] = (df["time_window_start"] - df["accept_time"]).dt.total_seconds() / 60
df["time_window_duration"] = (df["time_window_end"] - df["time_window_start"]).dt.total_seconds() / 60

In [ ]:
# ====================== 5. HAUL TYPE (Fixed - removed target leakage) ======================
df['is_long_haul'] = (df['distance_km'] > 8).astype(int)
df['distance_bucket'] = pd.qcut(df['distance_km'], q=8, duplicates='drop', labels=False)

# estimated_travel_time là OK vì chỉ phụ thuộc distance_km, không dùng target
df['estimated_travel_time'] = df['distance_km'] * 4   # ~15km/h

df['is_processing_dominant'] = (df['distance_km'] < 1.5).astype(int)

# ĐÃ XÓA: processing_time (= eta_minutes - estimated_travel_time) -> chứa target trực tiếp

In [ ]:
# ====================== 6. COURIER & AOI STATISTICS ======================
df["aoi_frequency"] = df.groupby("aoi_id")["aoi_id"].transform("count")
df["courier_frequency"] = df.groupby("courier_id")["courier_id"].transform("count")

# ⚠️ Vẫn leak (tính trên full data) — sẽ được tính lại đúng cách trong Step 5 sau split.
# Giữ tạm ở đây chỉ để tham khảo/EDA, KHÔNG dùng trực tiếp các cột _mean/_std này khi train.
courier_stats = df.groupby("courier_id").agg({
    'eta_minutes': ['mean', 'std'],
    'distance_km': 'mean'
}).reset_index()
courier_stats.columns = ['courier_id', 'courier_eta_mean', 'courier_eta_std',
                        'courier_dist_mean']

df = df.merge(courier_stats, on='courier_id', how='left')
# ĐÃ XÓA: 'processing_time': 'mean' -> courier_proc_mean (leak kép: vừa từ processing_time, vừa full-data)

In [ ]:
# ====================== 7. CLUSTERING ======================
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

coords = df[['lat', 'lng']]
scaler = StandardScaler()
coords_scaled = scaler.fit_transform(coords)

kmeans = KMeans(n_clusters=60, random_state=42, n_init=10)
df['coord_cluster'] = kmeans.fit_predict(coords_scaled)

In [ ]:
# ====================== 8. INTERACTIONS & FINAL CLEANING ======================
df['long_haul_rush'] = df['is_long_haul'] * df['is_peak_hour']
df['distance_x_hour'] = df['distance_km'] * df['accept_hour']
# ĐÃ XÓA: processing_x_rush (= processing_time * is_peak_hour) -> leak

df = df[df['eta_minutes'] <= df['eta_minutes'].quantile(0.99)].copy()

In [ ]:
# ====================== 9. FINAL FEATURE LIST ======================
num_cols = [
    'lat', 'lng', 'distance_km', 'estimated_travel_time',
    'accept_hour', 'accept_minute', 'day_of_week',
    'hour_sin', 'hour_cos',
    'time_to_window_start', 'time_window_duration',
    'aoi_frequency', 'courier_frequency',
    # courier_eta_mean/std, courier_dist_mean: KHÔNG dùng bản full-data này,
    # Step 5 sẽ tính lại sau split và override
    'distance_x_hour'
    # ĐÃ XÓA: 'processing_time', 'courier_proc_mean', 'processing_x_rush'
]

cat_cols = [
    'region_id', 'aoi_id', 'aoi_type',   # ĐÃ XÓA 'coord_cluster' (full-data fit, leak nhẹ) -> Step 5 dùng coord_cluster_v2
    'is_weekend', 'is_peak_hour', 'is_morning_rush', 'is_evening_rush',
    'is_long_haul', 'is_processing_dominant', 'gps_both'
]

print(f"Numerical features : {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)}")

Numerical features : 14
Categorical features: 10


In [ ]:
# ====================== 10. SAVE v4 ======================
output_path = f"{SAVE_DIR}/features_engineered_v4.parquet"
df.to_parquet(output_path, index=False)
print(f"Saved at: {output_path} | Shape: {df.shape}")

Saved at: /content/drive/MyDrive/LaDe/LaDe-P/features_engineered_v4.parquet | Shape: (1132429, 48)
